# 02 · Visualization: maps from the CRAF'd API

**What this teaches:** how to turn grid-level slices into PRIO-GRID maps: the whole world for one month, one country, several months on a shared scale, the three outcomes side by side, and the recorded deaths next to the forecast.

Grid rows carry each cell's centre as `latitude` and `longitude`. The maps use `views_crafdapi.plotting`. **You need an API key** (see `notebooks/README.md`).

## 1 · Setup

In [ ]:
import io
import os

import pandas as pd
import requests
from dotenv import load_dotenv

import matplotlib.pyplot as plt

from views_crafdapi.client import CrafdApiClient
from views_crafdapi.plotting import add_shared_colorbar, make_subplot_grid, plot_pixel_grid, shared_norm_from_dfs

load_dotenv()  # reads APPWRITE_DATASTORE_API_KEY (your key) and, optionally, VIEWS_API_URL

BASE_URL = os.environ.get("VIEWS_API_URL", "https://crafdapi.viewsforecasting.org")
API_KEY = os.environ.get("APPWRITE_DATASTORE_API_KEY", "").strip().strip('"')
HEADERS = {"X-API-Key": API_KEY}  # for the bulk download, which is a plain file request

client = CrafdApiClient(BASE_URL, API_KEY, timeout=300)
print("Base URL:", BASE_URL)
print("API key :", "set" if API_KEY else "MISSING: set it in .env")


def download_bulk(level, outcome):
    """One bulk file as a DataFrame, and the month the forecast was made in. A large file over a
    slow connection can time out or drop; one retry covers that."""
    for attempt in (1, 2):
        try:
            resp = requests.get(f"{BASE_URL}/bulk/{level}/{outcome}", headers=HEADERS, timeout=600)
            break
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError):
            if attempt == 2:
                raise
            print("The download was cut off; trying once more...")
    if not resp.ok:   # show the API's own explanation, not only the status code
        raise RuntimeError(f"API error {resp.status_code} on /bulk/{level}/{outcome}: {resp.text[:300]}")
    return pd.read_parquet(io.BytesIO(resp.content)), resp.headers["X-Forecast-Made-In"]

In [ ]:
# Stop early, with a clear message, when there is no usable key: otherwise the first data call
# fails with a 401 that does not say the .env is missing.
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit(
        "No usable API key. Copy .env.example to .env at the root of this repository, set\n"
        "APPWRITE_DATASTORE_API_KEY to your key (ask VIEWS for one), restart the kernel and\n"
        "run again. 03_offline_demo.ipynb needs no key."
    )

# /health also asks the key store; if only that fails, the data may still be served, so go on.
try:
    health = client.health()
except RuntimeError as e:
    health = {}
    print("Health check unavailable, continuing:", e)
else:
    print("Service:", health["status"], "| forecast loaded:", health.get("forecast_loaded"))
if health.get("forecast_loaded") is False:
    raise SystemExit("The service has no forecast loaded yet (it answers 503 to data requests). "
                     "Try again in a few minutes.")

In [ ]:
# The months and Nigeria's GAUL country code, from the small country bulk file (a slice of every
# month of every country would read too much and answer 413: that is what bulk files are for).
countries, MADE_IN = download_bulk("country", "sb")
FORECAST_MONTHS = sorted(countries.loc[countries["month"] >= MADE_IN, "month"].unique())
RECORDED_MONTHS = sorted(countries.loc[countries["month"] < MADE_IN, "month"].unique())
FIRST = FORECAST_MONTHS[0]
NIGERIA = int(countries.loc[countries["country_iso3"] == "NGA", "country_code"].iloc[0])
print("First forecast month:", FIRST, "| last recorded month:", RECORDED_MONTHS[-1])

## 2 · The whole world, one month

The expected number of state-based deaths per grid cell in the first forecast month, on a log scale.

In [ ]:
world = client.slice("grid", "sb", months=[FIRST], columns=["sb_mean"])
# Nigeria's grid cells, asked for by id below: smaller answers than the whole grid.
NGA_CELLS = world.loc[world["country_code"] == NIGERIA, "priogrid_id"].astype(int).tolist()
fig, ax, _ = plot_pixel_grid(world, lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                             transform="log1p", cmap="YlOrRd", figsize=(14, 7))
ax.set_title(f"Expected state-based deaths, {FIRST} (log scale)");

## 3 · One country: expected deaths and the chance of more than 25

The two maps answer different questions: how many deaths to expect, and how likely a serious month is. A chance is between 0 and 1, so it is drawn without the log scale.

In [ ]:
nga = client.slice("grid", "sb", places=NGA_CELLS, months=[FIRST], columns=["sb_mean", "sb_p_gt25"])
fig, axes = make_subplot_grid(2, cols=2, figsize_per_plot=(7, 7))
plot_pixel_grid(nga, ax=axes[0], lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                transform="log1p", cmap="YlOrRd")
axes[0].set_title(f"Nigeria: expected deaths, {FIRST} (log scale)")
plot_pixel_grid(nga, ax=axes[1], lon_col="longitude", lat_col="latitude", value_col="sb_p_gt25",
                transform="none", cmap="Purples")
axes[1].set_title(f"Nigeria: chance of more than 25 deaths, {FIRST}");

## 4 · Several forecast months on one scale

A shared colour scale makes the months comparable: a darker cell is a higher value in every panel.

In [ ]:
months = FORECAST_MONTHS[:3]
panel = client.slice("grid", "sb", places=NGA_CELLS, months=months, columns=["sb_mean"])
by_month = [panel[panel["month"] == m] for m in months]
norm = shared_norm_from_dfs(by_month, value_col="sb_mean", transform="log1p")
fig, axes = make_subplot_grid(len(months), cols=3, figsize_per_plot=(6, 6))
for ax, m, df in zip(axes, months, by_month):
    plot_pixel_grid(df, ax=ax, lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                    transform="log1p", cmap="YlOrRd", norm=norm, colorbar=False)
    ax.set_title(m)
add_shared_colorbar(fig, axes, norm=norm, cmap="YlOrRd", label="expected deaths (log1p)");

## 5 · The three outcomes side by side

State-based (`sb`), non-state (`ns`) and one-sided violence (`os`) are forecast separately; each is its own slice with its own columns. Each panel has its own colour scale here, because the three outcomes differ a lot in size: compare patterns, not shades.

In [ ]:
labels = {"sb": "State-based", "ns": "Non-state", "os": "One-sided"}
fig, axes = make_subplot_grid(3, cols=3, figsize_per_plot=(6, 6))
for ax, (outcome, label) in zip(axes, labels.items()):
    df = client.slice("grid", outcome, places=NGA_CELLS, months=[FIRST], columns=[f"{outcome}_mean"])
    plot_pixel_grid(df, ax=ax, lon_col="longitude", lat_col="latitude", value_col=f"{outcome}_mean",
                    transform="log1p", cmap="YlOrRd")
    ax.set_title(f"{label}, {FIRST}")

## 6 · Recorded deaths next to the forecast

The last recorded month (`sb_actual`, a count) beside the first forecast month (`sb_mean`, an expectation). The recorded month is one month before the forecast starts: the first forecast month itself is barely reported when the forecast is made. The two panels have their own colour scales: a recorded count and an expectation differ in kind.

In [ ]:
last = RECORDED_MONTHS[-1]
recorded = client.slice("grid", "sb", places=NGA_CELLS, months=[last], columns=["sb_actual"])
fig, axes = make_subplot_grid(2, cols=2, figsize_per_plot=(7, 7))
plot_pixel_grid(recorded, ax=axes[0], lon_col="longitude", lat_col="latitude", value_col="sb_actual",
                transform="log1p", cmap="Greys")
axes[0].set_title(f"Recorded deaths, {last}")
plot_pixel_grid(nga, ax=axes[1], lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                transform="log1p", cmap="YlOrRd")
axes[1].set_title(f"Expected deaths, {FIRST}");

## Next steps

- **`01_quickstart.ipynb`**: the bulk files, slices and the summary columns.
- **`03_offline_demo.ipynb`**: the analytics on synthetic data, no key needed.
- The [data dictionary](../docs/api/data_dictionary.md) explains every column.